In [1]:
import pandas as pd

df = pd.read_csv("nhanes_cardiometabolic_subset_dataset.csv")

print("Original dataset shape:", df.shape)

Original dataset shape: (10175, 64)


In [3]:
bp_vars = [
    "BPXSY1","BPXSY2","BPXSY3","BPXSY4",
    "BPXDI1","BPXDI2","BPXDI3","BPXDI4",
    "BPXSY_AVG","BPXDI_AVG",
    "BPXPULS",
    "BPQ020","BPQ030","BPQ080",
    "BP_CATEGORY","HTN_STAGE"
]

df = df.drop(columns=bp_vars, errors="ignore")

print("Dataset shape after removing BP variables:", df.shape)

Dataset shape after removing BP variables: (10175, 48)


In [5]:
target = "HYPERTENSION_YN"

X = df.drop(columns=[target])
y = df[target]

print("Total predictors:", X.shape[1])

Total predictors: 47


In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("Training shape:", X_train.shape)
print("Test shape:", X_test.shape)

Training shape: (8140, 47)
Test shape: (2035, 47)


In [9]:
numeric_features = X_train.select_dtypes(include=["int64","float64"]).columns.tolist()
categorical_features = X_train.select_dtypes(include=["object","category"]).columns.tolist()

print("Numeric predictors:", len(numeric_features))
print("Categorical predictors:", len(categorical_features))

Numeric predictors: 47
Categorical predictors: 0


In [11]:
# convert categorical variables

categorical_vars = [
    "RIAGENDR",
    "RIDRETH1",
    "DMDEDUC2",
    "DMDMARTL",
    "DIQ010",
    "DIQ050",
    "SMQ020",
    "PAQ605",
    "PAQ620",
    "PAQ650",
    "DBQ700",
    "SLQ050",
    "SLQ060",
    "HSD010",
    "BPQ080"
]

for col in categorical_vars:
    if col in df.columns:
        df[col] = df[col].astype("category")

print("Categorical conversion complete.")

Categorical conversion complete.


In [17]:
X = df.drop(columns=[target])
y = df[target]

numeric_features = X.select_dtypes(include=["int64","float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["category","object"]).columns.tolist()

print("Numeric predictors:", len(numeric_features))
print("Categorical predictors:", len(categorical_features))

Numeric predictors: 33
Categorical predictors: 14


In [19]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessing = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

In [21]:
from sklearn.linear_model import LogisticRegression

log_model = Pipeline([
    ("preprocess", preprocessing),
    ("model", LogisticRegression(max_iter=1000))
])

log_model.fit(X_train, y_train)

Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['SEQN', 'BPQ050A',
                                                   'RIDAGEYR', 'INDFMPIR',
                                                   'BMXWT', 'BMXHT', 'BMXBMI',
                                                   'BMXWAIST', 'BMXARML',
                                                   'BMXARMC', 'BMXLEG', 'LBXGH',
                                                   'LBXIN', 'DIQ070', 'LBXTC',
                                                   'LBDHDD', 'LBDLDL', 'LBXTR',
                                                   'RATIO_TC_HDL',
                                                   'RATIO_TG_H...
                                                   'LBXWBCSI', 'LBXPLTSI',
                                                   'SMQ040', 'ALQ120Q', ...]),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['RIAGENDR', 'RIDRETH1',
                                                   'DMDEDUC2', 'DMDMARTL',
                                                   'DIQ010', 'DIQ050', 'SMQ020',
                                                   'PAQ605', 'PAQ620', 'PAQ650',
                                                   'DBQ700', 'SLQ050', 'SLQ060',
                                                   'HSD010'])])),
                ('model', LogisticRegression(max_iter=1000))])

In [27]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

y_pred_log = log_model.predict(X_test)

print("Logistic Regression Results")
print("Accuracy:", accuracy_score(y_test, y_pred_log))
print("Precision:", precision_score(y_test, y_pred_log))
print("Recall:", recall_score(y_test, y_pred_log))
print("F1 Score:", f1_score(y_test, y_pred_log))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_log))

Logistic Regression Results
Accuracy: 0.8486486486486486
Precision: 0.7533333333333333
Recall: 0.738562091503268
F1 Score: 0.7458745874587459

Confusion Matrix:
[[1275  148]
 [ 160  452]]


In [29]:
from sklearn.ensemble import RandomForestClassifier

rf_model = Pipeline([
    ("preprocess", preprocessing),
    ("model", RandomForestClassifier(
        n_estimators=200,
        random_state=42
    ))
])

rf_model.fit(X_train, y_train)

Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['SEQN', 'BPQ050A',
                                                   'RIDAGEYR', 'INDFMPIR',
                                                   'BMXWT', 'BMXHT', 'BMXBMI',
                                                   'BMXWAIST', 'BMXARML',
                                                   'BMXARMC', 'BMXLEG', 'LBXGH',
                                                   'LBXIN', 'DIQ070', 'LBXTC',
                                                   'LBDHDD', 'LBDLDL', 'LBXTR',
                                                   'RATIO_TC_HDL',
                                                   'RATIO_TG_H...
                                                   'SMQ040', 'ALQ120Q', ...]),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['RIAGENDR', 'RIDRETH1',
                                                   'DMDEDUC2', 'DMDMARTL',
                                                   'DIQ010', 'DIQ050', 'SMQ020',
                                                   'PAQ605', 'PAQ620', 'PAQ650',
                                                   'DBQ700', 'SLQ050', 'SLQ060',
                                                   'HSD010'])])),
                ('model',
                 RandomForestClassifier(n_estimators=200, random_state=42))])

In [33]:
y_pred_rf = rf_model.predict(X_test)

print("Random Forest Results")
print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print("Precision:", precision_score(y_test, y_pred_rf))
print("Recall:", recall_score(y_test, y_pred_rf))
print("F1 Score:", f1_score(y_test, y_pred_rf))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf))

Random Forest Results
Accuracy: 0.8471744471744471
Precision: 0.7385103011093502
Recall: 0.761437908496732
F1 Score: 0.7497988736926791

Confusion Matrix:
[[1258  165]
 [ 146  466]]


In [35]:
from sklearn.model_selection import cross_val_score

auc_scores = cross_val_score(
    rf_model,
    X,
    y,
    cv=5,
    scoring="roc_auc"
)

print("Cross-validation AUC scores:", auc_scores)
print("Mean AUC:", auc_scores.mean().round(3))
print("Std Dev:", auc_scores.std().round(3))

Cross-validation AUC scores: [0.92613989 0.91846256 0.9222306  0.91081336 0.92550891]
Mean AUC: 0.921
Std Dev: 0.006
